In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import statsmodels.api as sm 
from statsmodels.formula.api import ols
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator

In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'

In [ ]:
df = pd.read_excel(fpath + '\\4.0_database_variables.xlsx')

In [ ]:
df = df.rename(columns={'location(ita=0,uk=1,usa=2)': 'location', 'weekday(1=free days)': 'weekday'})

In [ ]:
# remove outliers
df = df[(np.abs(stats.zscore(df['sleep_duration(h)'])) < 3)]
df = df[(np.abs(stats.zscore(df['midpoint_h'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_start_decimal'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_end_decimal'])) < 3)]
df = df[(np.abs(stats.zscore(df['phase(sleepoffset-sunrise)'])) < 3)]

In [ ]:
df = df.drop('sunrise_time(USA)', axis=1)
df = df.drop('sunrise (USA), hours', axis=1)
df = df.drop('sunset (USA), hours', axis=1)
df = df.drop('sunset_time(USA)', axis=1)
df = df.drop('photoperiod (h, USA)', axis=1)

In [ ]:
# Define the start date
start_date = pd.to_datetime('2022-09-21')

In [ ]:
# Function to calculate the week of the year from the start date
def calculate_week_of_year(date):
    year_diff = date.year - start_date.year
    start_of_year = pd.to_datetime(f'{date.year}-01-01')
    weeks_from_start = ((date - start_of_year).days // 7) + 1
    return year_diff * 52 + weeks_from_start

# Apply the function to calculate the week of the year
df['week_of_year'] = df['date'].apply(calculate_week_of_year)

In [ ]:
# adjust 'week of the year' to start from 0
df['week_of_year'] = df['week_of_year'] - 37

In [ ]:
# rename the location column as 0=ITA, 1=UK
df['location'] = df['location'].map({0: 'ITA', 1: 'UK'})

# rename the weekday column as 0=work days, 1=free days
df['weekday'] = df['weekday'].map({0: 'work days', 1: 'free days'})

# Descriptive analysis

In [ ]:
all_descriptive = df.describe()
all_descriptive = all_descriptive.transpose()

In [ ]:
# test normality of the data using Shapiro-Wilk test 
# H0: data is normally distributed
shapiro_test_sleep_duration = stats.shapiro(df['sleep_duration(h)'])
shapiro_test_midpoint = stats.shapiro(df['midpoint_h'])
shapiro_test_sleep_start = stats.shapiro(df['sleep_start_decimal'])
shapiro_test_sleep_end = stats.shapiro(df['sleep_end_decimal'])
shapiro_test_phase = stats.shapiro(df['phase(sleepoffset-sunrise)'])

In [ ]:
shapiro_results_descriptive = pd.DataFrame({
    'Variable': ['sleep_duration(h)', 'midpoint_h', 'sleep_start_decimal', 'sleep_end_decimal', 'phase(sleepoffset-sunrise)'],
    'Shapiro-Wilk test': [shapiro_test_sleep_duration, shapiro_test_midpoint, shapiro_test_sleep_start, shapiro_test_sleep_end, shapiro_test_phase]
})

In [ ]:
shapiro_results_descriptive

In [ ]:
df_grouped_location = df.groupby('location').describe()
df_grouped_location = df_grouped_location.transpose()

In [ ]:
df_grouped_weekday = df.groupby('weekday').describe()
df_grouped_weekday = df_grouped_weekday.transpose()

In [ ]:
df_grouped_location_weekday = df.groupby(['location', 'weekday']).describe()
df_grouped_location_weekday = df_grouped_location_weekday.transpose()

In [ ]:
# filtered the midpoints by week days
df_workdays = df[df['weekday'] == 'work days']
df_freedays = df[df['weekday'] == 'free days']

In [ ]:
# compare the midpoint, duration and phase between the two locations
ttest_midpoint_workdays_loc = stats.ttest_ind(df_workdays[df_workdays['location'] == 'ITA']['midpoint_h'], df_workdays[df_workdays['location'] == 'UK']['midpoint_h'])
ttest_midpoint_freedays_loc = stats.ttest_ind(df_freedays[df_freedays['location'] == 'ITA']['midpoint_h'], df_freedays[df_freedays['location'] == 'UK']['midpoint_h'])
utest_duration_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_duration(h)'], df[df['location'] == 'UK']['sleep_duration(h)'])
utest_phase_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['phase(sleepoffset-sunrise)'], df[df['location'] == 'UK']['phase(sleepoffset-sunrise)'])


# compare the midpoint, duration and phase between work days and free days
ttest_midpoint_week = stats.ttest_ind(df[df['weekday'] == 'work days']['midpoint_h'], df[df['weekday'] == 'free days']['midpoint_h'])
utest_duration_week = stats.mannwhitneyu(df[df['weekday'] == 'work days']['sleep_duration(h)'], df[df['weekday'] == 'free days']['sleep_duration(h)'])
utest_phase_week = stats.mannwhitneyu(df[df['weekday'] == 'work days']['phase(sleepoffset-sunrise)'], df[df['weekday'] == 'free days']['phase(sleepoffset-sunrise)'])

In [ ]:
print('Results for location')
print('Midpoint_work:', ttest_midpoint_workdays_loc)
print('Midpoint_free:', ttest_midpoint_freedays_loc)
print('Sleep_duration:', utest_duration_loc)
print('Phase:', utest_phase_loc)

In [ ]:
print('Results for weekday') 
print('Midpoint:', ttest_midpoint_week)
print('Sleep _duration:', utest_duration_week)
print('Phase:', utest_phase_week)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

sns.boxplot(x='location', y='midpoint_h', data=df_workdays, ax=ax[0])
ax[0].set_title('Midpoint of sleep (work days) by location')
ax[0].set_ylabel('Midpoint (hours)')
ax[0].set_xlabel('')
ax[0].yaxis.set_major_locator(MultipleLocator(1))
ax[0].set_ylim(23, 31)
#add a significance line of ** for the p-value < 0.01
if ttest_midpoint_workdays_loc.pvalue < 0.01:
    ax[0].annotate('**', xy=(0.5, 0.9), xycoords='axes fraction', ha='center', va='center', fontsize=12) 

sns.boxplot(x='location', y='midpoint_h', data=df_freedays, ax=ax[1])
ax[1].set_title('Midpoint of sleep (free days) by location')
ax[1].set_ylabel('Midpoint (hours)')
ax[1].set_xlabel('')
ax[1].yaxis.set_major_locator(MultipleLocator(1))
ax[1].set_ylim(ax[0].get_ylim())
#add a significance line of  for the p-value < 0.05
if ttest_midpoint_workdays_loc.pvalue < 0.01:
    ax[1].annotate('', xy=(0.5, 0.9), xycoords='axes fraction', ha='center', va='center', fontsize=12) 
    
sns.despine()
plt.grid(False)
plt.gca().spines['bottom'].set_color('black') # set the color of the x-axis to black
plt.gca().spines['left'].set_color('black')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
sns.boxplot(x='location', y='sleep_duration(h)', data=df, gap=0.2)
plt.title('Mean of sleep duration by location')
plt.xlabel('')
plt.ylabel('Sleep duration (h)')
sns.despine()
plt.grid(False)
plt.gca().spines['bottom'].set_color('black') # set the color of the x-axis to black
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
plt.figure()
sns.boxplot(x='location', y='phase(sleepoffset-sunrise)', data=df, gap=0.2)
plt.title('Mean of phase by location')
plt.xlabel('location')
plt.ylabel('phase(sleepoffset-sunrise)')
sns.despine()
plt.grid(False)
plt.gca().spines['bottom'].set_color('black') # set the color of the x-axis to black
plt.gca().spines['left'].set_color('black')

#add a significance line 
if utest_phase_loc.pvalue < 0.001:
    plt.annotate('***', xy=(0.5, 0.9), xycoords='axes fraction', ha='center', fontsize=12)
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='phase(sleepoffset-sunrise)', hue='location', data=df)
plt.title('Phase by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Phase (sleep offset - sunrise)')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())

# get the current axes then set the x-axis major locator
# plot.gca()=ax if set ax = plt.gca()
plt.gca().xaxis.set_major_locator(MultipleLocator(30)) 

#add the mean of the phase for the two locations
plt.axhline(df[df['location'] == 'ITA']['phase(sleepoffset-sunrise)'].mean(), color='#00589c', linestyle='--', label='ITA mean')
plt.axhline(df[df['location'] == 'UK']['phase(sleepoffset-sunrise)'].mean(), color='#ec611a', linestyle='--', label='UK mean')

plt.show()

_Jetlag_

In [ ]:
data_jetlag = df # create a new dataframe with the original data

In [ ]:
#download the data
data_jetlag.to_csv(fpath + '\\jetlag_data.csv', index=False)

In [ ]:
# calculate the mean midpoint for each location, week and weekday
weekly_means_jetlag = data_jetlag.groupby(['location', 'week_of_year', 'weekday'])['midpoint_h'].mean().unstack()

In [ ]:
weekly_means_jetlag['jet lag'] = weekly_means_jetlag['free days'] - weekly_means_jetlag['work days']

In [ ]:
# add a column with the location
weekly_means_jetlag['location'] = weekly_means_jetlag.index.get_level_values(0)

In [ ]:
# Plot the weekly jet lag between UK and ITA
plt.figure(figsize=(12, 6))
sns.lineplot(x='week_of_year', y='jet lag', hue='location', data=weekly_means_jetlag, marker='o', palette=['darkblue', 'darkorange'])
plt.title('Weekly Jet Lag Comparison between UK and ITA (from 21-09-2022)')
plt.xlabel('Weeks counted')
plt.ylabel('Jet Lag (hours)')
plt.legend(title='Location', labels=['ITA', 'UK'], loc='upper right', handles=[plt.Line2D([0], [0], color='darkblue', lw=2), plt.Line2D([0], [0], color='darkorange', lw=2)])
plt.grid(True)
plt.xlim(0, 104)
plt.gca().xaxis.set_major_locator(MultipleLocator(4)) 
plt.xticks()
plt.show()

In [ ]:
# Remove NaN values before performing the Shapiro-Wilk test
jetlag_no_nan = weekly_means_jetlag['jet lag'].dropna()

In [ ]:
# Test normality of the jet lag data using Shapiro-Wilk test 
# H0: data is normally distributed
shapiro_test_jetlag = stats.shapiro(jetlag_no_nan)

In [ ]:
shapiro_test_jetlag

In [ ]:
# Perform a t-test to compare the jet lag between the two locations
ttest_jetlag = stats.ttest_ind(weekly_means_jetlag[weekly_means_jetlag['location'] == 'ITA']['jet lag'].dropna(), weekly_means_jetlag[weekly_means_jetlag['location'] == 'UK']['jet lag'].dropna())

print('Results for jet lag:')
print(ttest_jetlag)

In [ ]:
# plot the jet lag
plt.figure()
sns.boxplot(x='location', y='jet lag', data=weekly_means_jetlag)
plt.title('Jet lag by location')
plt.xlabel('')
plt.ylabel('Jet lag, h')

sns.despine()
plt.grid(False)
# bottom and left spine black
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')

plt.show()

_Seasonal effect_

In [ ]:
df = df.rename(columns={'sleep_duration(h)': 'sleep_duration'})

In [ ]:
df = df.rename(columns={'phase(sleepoffset-sunrise)': 'phase'})

In [ ]:
# Adding a 'season' column to the dataset based on the 'date' column
# Defining seasons based on months: 
# Winter (Dec-Feb), Spring (Mar-May), Summer (Jun-Aug), Autumn (Sep-Nov)

def assign_season(date):
    month = pd.to_datetime(date).month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

In [ ]:
# Applying the function to create a season column
df_workdays['season'] = df_workdays['date'].apply(assign_season)
df_freedays['season'] = df_freedays['date'].apply(assign_season)
df['season'] = df['date'].apply(assign_season)

In [ ]:
df = df.dropna(subset=['sleep_duration'])
df = df.dropna(subset=['phase'])

In [ ]:
anova_midpoint_work_season1 = ols('midpoint_h ~ C(season)', data=df_workdays).fit()

In [ ]:
anova_midpoint_free_season1 = ols('midpoint_h ~ C(season)', data=df_freedays).fit()

In [ ]:
anova_midpoint_work_result_season1 = sm.stats.anova_lm(anova_midpoint_work_season1, typ=3) #fit the ANOVA model and get the results

In [ ]:
anova_midpoint_free_result_season1 = sm.stats.anova_lm(anova_midpoint_free_season1, typ=3)

In [ ]:
print("\nANOVA_midpoint work Results:")
print(anova_midpoint_work_result_season1)
print("\nANOVA_midpoint free Results:")
print(anova_midpoint_free_result_season1)

In [ ]:
# Kruskal-Wallis test 
kw_sleep_duration_season = stats.kruskal(df[df['season'] == 'Winter']['sleep_duration'], df[df['season'] == 'Spring']['sleep_duration'], df[df['season'] == 'Summer']['sleep_duration'], df[df['season'] == 'Autumn']['sleep_duration'])
kw_phase_season = stats.kruskal(df[df['season'] == 'Winter']['phase'], df[df['season'] == 'Spring']['phase'], df[df['season'] == 'Summer']['phase'], df[df['season'] == 'Autumn']['phase'])

# Print the results of the Kruskal-Wallis test
print('Results for Kruskal-Wallis test for the sleep duration by season')
print(kw_sleep_duration_season)
print('Results for Kruskal-Wallis test for the phase by season')
print(kw_phase_season)


In [ ]:
# perform a Tukey HSD test to compare the means
tukey_results_season1 = pairwise_tukeyhsd(df_workdays['midpoint_h'], df_workdays['season'])
print(tukey_results_season1)

In [ ]:
# perform a Tukey HSD test to compare the means
tukey_results_season2 = pairwise_tukeyhsd(df_freedays['midpoint_h'], df_freedays['season'])
print(tukey_results_season2)

In [ ]:
# Perform a Tukey HSD test to compare the means 
tukey_results_season3 = pairwise_tukeyhsd(df['sleep_duration'], df['season'])
print(tukey_results_season3)

In [ ]:
# Perform a Tukey HSD test to compare the means
tukey_results_season4 = pairwise_tukeyhsd(df['phase'], df['season'])
print(tukey_results_season4)

In [ ]:
# Sleep Midpoint (work) by season
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='midpoint_h', data=df_workdays)
plt.title('Sleep Midpoint (workdays) by season')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Midpoint of Sleep (hours)')
plt.xlabel('')
sns.despine()
plt.grid(False)
#bottom and left spine black
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# Sleep Midpoint (free) by season
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='midpoint_h', data=df_freedays)
plt.title('Sleep Midpoint (freedays) by season')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Midpoint of Sleep (hours)')
plt.xlabel('')
sns.despine()
plt.grid(False)
# bottom and left spine black
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# Sleep Duration by season
plt.figure(figsize=(12, 6))
df.boxplot(column='sleep_duration', by='season', grid=False)
plt.title('Sleep Duration by season')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Sleep Duration (hours)')
plt.xlabel('Season')
sns.despine()
plt.grid(False)
# bottom and left spine black
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# Sleep Phase by season
plt.figure(figsize=(12, 6))
df.boxplot(column='phase', by='season', grid=False)
plt.title('Sleep Phase by Season')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Phase (sleep offset - sunrise)')
plt.xlabel('Season')
sns.despine()
plt.grid(False)
# bottom and left spine black
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

_Phase by location and time_

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
data_phase = df.dropna(subset=['phase', 'location', 'date'])

In [ ]:
data_phase['location'] = data_phase['location'].map({'ITA': 0, 'UK': 1})

In [ ]:
# Converting date to numerical values (days since the start of the observation period)
data_phase['date_numeric'] = (pd.to_datetime(data_phase['date']) - pd.to_datetime(data_phase['date']).min()).dt.days

In [ ]:
glm_model_phase_numeric = smf.glm(
    formula="phase ~ location + date_numeric",
    data=data_phase,
    family=sm.families.Gaussian()
)

In [ ]:
# Fit of the model
glm_results_phase_numeric = glm_model_phase_numeric.fit()

In [ ]:
# Output the summary of the model
glm_results_phase_numeric_summary = glm_results_phase_numeric.summary()
glm_results_phase_numeric_summary

_Midpoint by location and week day_

In [ ]:
model_1 = ols('midpoint_h ~ C(location) * C(weekday)', data=df).fit() # C() is used to indicate categorical variables

In [ ]:
anova_results = sm.stats.anova_lm(model_1, typ=3)

In [ ]:
print("ANOVA Results:")
print(anova_results)

In [ ]:
# coefficients of the model
weights = model_1.params / model_1.params.abs().sum()
weights

In [ ]:
# post-hoc test 
posthoc_midpoint = pairwise_tukeyhsd(
    endog=df['midpoint_h'],  
    groups=df['location'].astype(str) + '_' + df['weekday'].astype(str),  
    alpha=0.05  
)

In [ ]:
print(posthoc_midpoint)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='location', y='midpoint_h', hue='weekday', data=df, gap=0.2)
plt.title('Mean of midpoint by location and weekday')
plt.xlabel('Location')
plt.ylim(23, 33)
plt.ylabel('Midpoint (h)')
plt.legend(title='Weekday')
plt.show()

_Sleep duration by location and week day_

In [ ]:
data_sleep_duration = df.dropna(subset=['sleep_duration', 'location', 'weekday'])

In [ ]:
glm_model_sleep_duration = smf.glm(
    formula="sleep_duration ~ location + weekday",
    data=data_sleep_duration,
    family=sm.families.Gaussian()
)

In [ ]:
# Fit of the model
glm_results_sleep_duration = glm_model_sleep_duration.fit()

In [ ]:
# Output the summary of the model (ita and free days as baseline)
glm_results_sleep_duration_summary = glm_results_sleep_duration.summary()
glm_results_sleep_duration_summary

In [ ]:
# post-hoc test 
posthoc_sleep_duration = pairwise_tukeyhsd(
    endog=df['sleep_duration'],  
    groups=data_sleep_duration['location'].astype(str) + '_' + data_sleep_duration['weekday'].astype(str),  
    alpha=0.05  
)

In [ ]:
print(posthoc_sleep_duration)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='location', y='sleep_duration', hue='weekday', data=data_sleep_duration, gap=0.2)
plt.title('Mean of sleep duration by location and weekday')
plt.xlabel('Location')
plt.ylim()
plt.ylabel('Sleep duration (h)')
plt.legend(title='Weekday')
plt.show()

_Phase by location and week day_

In [ ]:
data_phase2 = df.dropna(subset=['phase', 'location', 'weekday'])

In [ ]:
glm_model_phase2 = smf.glm(
    formula="phase ~ location + weekday",
    data=data_phase2,
    family=sm.families.Gaussian()
)

In [ ]:
# Fit of the model
glm_results_phase2 = glm_model_phase2.fit()

In [ ]:
# Output the summary of the model (ita and free days as baseline)
glm_results_phase2_summary = glm_results_phase2.summary()
glm_results_phase2_summary

In [ ]:
# post-hoc test 
posthoc_3 = pairwise_tukeyhsd(
    endog=data_phase2['phase'],  
    groups=data_phase2['location'].astype(str) + '_' + data_phase2['weekday'].astype(str),  
    alpha=0.05  
)

In [ ]:
print(posthoc_3)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='location', y='phase', hue='weekday', data=data_phase2, gap=0.2)
plt.title('Phase by location and weekday')
plt.xlabel('Location')
plt.ylim(-7,7)
plt.ylabel('Phase, h [sleep offset - sunrise]')
plt.legend(title='Weekday')

plt.show()

_Photoperiod and sleep-wake pattern_

In [ ]:
# Selecting relevant columns for correlation analysis
sleep_wake_vars = ["sleep_duration", "midpoint_h"]
photoperiod_vars = ["photoperiod (h, UK)", "photoperiod (h, ITA)"]

In [ ]:
# Calculating the correlation matrix between photoperiod and sleep-wake variables
correlation_matrix = data_sleep_duration[sleep_wake_vars + photoperiod_vars].corr()

In [ ]:
# Displaying the correlation matrix
correlation_matrix

In [ ]:
plt.figure(figsize=(12, 10))

# Plot 1: Photoperiod (ITA) vs Sleep Duration
plt.subplot(2, 2, 1)
sns.regplot(x="photoperiod (h, ITA)", y="sleep_duration", data=data_sleep_duration, scatter_kws={'alpha':0.5})
plt.title("Sleep Duration vs Photoperiod (ITA)")
plt.xlabel("Photoperiod (hours, ITA)")
plt.ylabel("Sleep Duration (hours)")

# Plot 2: Photoperiod (ITA) vs Midpoint of Sleep
plt.subplot(2, 2, 2)
sns.regplot(x="photoperiod (h, ITA)", y="midpoint_h", data=data_sleep_duration, scatter_kws={'alpha':0.5})
plt.title("Midpoint of Sleep vs Photoperiod (ITA)")
plt.xlabel("Photoperiod (hours, ITA)")
plt.ylabel("Midpoint of Sleep (hour)")

# Plot 3: Photoperiod (UK) vs Sleep Duration
plt.subplot(2, 2, 3)
sns.regplot(x="photoperiod (h, UK)", y="sleep_duration", data=data_sleep_duration, scatter_kws={'alpha':0.5})
plt.title("Sleep Duration vs Photoperiod (UK)")
plt.xlabel("Photoperiod (hours, UK)")
plt.ylabel("Sleep Duration (hours)")

# Plot 4: Photoperiod (UK) vs Midpoint of Sleep
plt.subplot(2, 2, 4)
sns.regplot(x="photoperiod (h, UK)", y="midpoint_h", data=data_sleep_duration, scatter_kws={'alpha':0.5})
plt.title("Midpoint of Sleep vs Photoperiod (UK)")
plt.xlabel("Photoperiod (hours, UK)")
plt.ylabel("Midpoint of Sleep (hour)")

# Display the plots
plt.tight_layout()
plt.show()
